# GitHub Repository Metrics - Complete Exploration & Star Prediction

**5,500 repositories with 29 features -- stars, forks, CI/CD, test coverage, and more**

---

> **TL;DR** -- This notebook provides an in-depth exploration of 5,500 synthetic GitHub repositories. We analyze language distributions, star/fork power-law patterns, correlation structures, CI/CD adoption rates, topic popularity, and license trends. We build a **star prediction model** (R-squared reported) using gradient boosting with feature importance analysis. Perfect for practicing regression, EDA, and feature engineering on software engineering data.

**Contents:**
1. [Data Overview](#1)
2. [Language Distribution](#2)
3. [Stars & Popularity Analysis](#3)
4. [Correlation Analysis](#4)
5. [Repository Health Metrics](#5)
6. [Topic Analysis](#6)
7. [License Analysis](#7)
8. [Sample ML Task: Star Prediction](#8)
9. [Key Insights & Next Steps](#9)

---

If you find this exploration useful, please **upvote the dataset and this notebook**!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

# Load data
import os
if os.path.exists('/kaggle/input/github-repo-metrics/github_repos.csv'):
    df = pd.read_csv('/kaggle/input/github-repo-metrics/github_repos.csv')
else:
    df = pd.read_csv('github_repos.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 1. Data Overview

In [ ]:
print('=== Basic Statistics ===')
print(f'Total repositories: {len(df):,}')
print(f'Languages: {df["language"].nunique()}')
print(f'Licenses: {df["license"].nunique()}')
print(f'\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\n=== Numeric Summary ===')
df[['stars', 'forks', 'open_issues', 'contributors', 'commits']].describe().round(1)

## 2. Language Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Repo count by language
lang_counts = df['language'].value_counts()
lang_counts.plot(kind='barh', ax=axes[0], color=plt.cm.tab20(np.linspace(0, 1, len(lang_counts))))
axes[0].set_title('Repositories per Language')
axes[0].set_xlabel('Count')

# Median stars by language
lang_stars = df.groupby('language')['stars'].median().sort_values(ascending=False)
lang_stars.plot(kind='barh', ax=axes[1], color='goldenrod')
axes[1].set_title('Median Stars by Language')
axes[1].set_xlabel('Median Stars')

plt.tight_layout()
plt.show()

## 3. Stars & Popularity Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Star distribution (log scale)
df['stars'].clip(lower=1).hist(bins=50, ax=axes[0, 0], color='gold', edgecolor='white', log=True)
axes[0, 0].set_title('Star Distribution (log scale)')
axes[0, 0].set_xlabel('Stars')
axes[0, 0].set_ylabel('Count (log)')

# Stars vs Forks
axes[0, 1].scatter(df['stars'], df['forks'], alpha=0.15, s=5, c='steelblue')
axes[0, 1].set_title('Stars vs. Forks')
axes[0, 1].set_xlabel('Stars')
axes[0, 1].set_ylabel('Forks')
axes[0, 1].set_xscale('symlog')
axes[0, 1].set_yscale('symlog')

# Stars vs Contributors
axes[1, 0].scatter(df['stars'], df['contributors'], alpha=0.15, s=5, c='coral')
axes[1, 0].set_title('Stars vs. Contributors')
axes[1, 0].set_xlabel('Stars')
axes[1, 0].set_ylabel('Contributors')
axes[1, 0].set_xscale('symlog')
axes[1, 0].set_yscale('symlog')

# Age vs Stars
df['created_date'] = pd.to_datetime(df['created_date'])
df['age_days'] = (pd.Timestamp('2025-01-01') - df['created_date']).dt.days
axes[1, 1].scatter(df['age_days'], df['stars'], alpha=0.15, s=5, c='green')
axes[1, 1].set_title('Repository Age vs. Stars')
axes[1, 1].set_xlabel('Age (days)')
axes[1, 1].set_ylabel('Stars')
axes[1, 1].set_yscale('symlog')

plt.tight_layout()
plt.show()

print('Star distribution percentiles:')
for p in [50, 75, 90, 95, 99]:
    print(f'  {p}th percentile: {df["stars"].quantile(p/100):,.0f} stars')

## 4. Correlation Analysis

In [ ]:
numeric_cols = ['stars', 'forks', 'watchers', 'open_issues', 'closed_issues',
                'contributors', 'commits', 'releases', 'readme_length',
                'has_ci', 'size_kb', 'age_days']

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right')
ax.set_yticklabels(numeric_cols)
plt.colorbar(im, shrink=0.8)
ax.set_title('Feature Correlation Matrix')

# Add correlation values
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        val = corr.iloc[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.tight_layout()
plt.show()

print('\nTop correlations with stars:')
star_corr = corr['stars'].drop('stars').abs().sort_values(ascending=False)
print(star_corr.to_string())

## 5. Repository Health Metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# CI adoption by language
ci_by_lang = df.groupby('language')['has_ci'].mean().sort_values(ascending=False)
ci_by_lang.plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('CI/CD Adoption by Language')
axes[0].set_xlabel('% with CI')

# Test coverage distribution (non-null only)
df['test_coverage'].dropna().hist(bins=30, ax=axes[1], color='mediumseagreen', edgecolor='white')
axes[1].set_title('Test Coverage Distribution (repos with CI)')
axes[1].set_xlabel('Coverage %')

# Archive rate by age
df['age_bucket'] = pd.cut(df['age_days'], bins=[0, 180, 365, 730, 1825, 3650],
                           labels=['<6mo', '6-12mo', '1-2yr', '2-5yr', '5+yr'])
archive_by_age = df.groupby('age_bucket')['is_archived'].mean()
archive_by_age.plot(kind='bar', ax=axes[2], color='tomato')
axes[2].set_title('Archive Rate by Repository Age')
axes[2].set_ylabel('% Archived')

plt.tight_layout()
plt.show()

print(f'\nOverall CI adoption: {df["has_ci"].mean():.1%}')
print(f'Overall archive rate: {df["is_archived"].mean():.1%}')
print(f'Repos with code of conduct: {df["has_code_of_conduct"].mean():.1%}')
print(f'Repos with contributing guide: {df["has_contributing_guide"].mean():.1%}')

## 6. Topic Analysis

In [ ]:
# Most common topics
topic_series = df['topics'].str.split('|').explode()
topic_counts = topic_series.value_counts().head(25)

fig, ax = plt.subplots(figsize=(12, 7))
topic_counts.plot(kind='barh', ax=ax, color=plt.cm.plasma(np.linspace(0.2, 0.8, len(topic_counts))))
ax.set_title('Top 25 Repository Topics')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

# Topics by median stars
topic_expanded = df.assign(topic=df['topics'].str.split('|')).explode('topic')
top_topic_stars = topic_expanded.groupby('topic')['stars'].median().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
top_topic_stars.plot(kind='barh', ax=ax, color='goldenrod')
ax.set_title('Top 15 Topics by Median Stars')
ax.set_xlabel('Median Stars')
plt.tight_layout()
plt.show()

## 7. License Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# License distribution
license_counts = df['license'].value_counts()
license_counts.plot(kind='bar', ax=axes[0], color=plt.cm.Set3(range(len(license_counts))))
axes[0].set_title('License Distribution')
axes[0].set_ylabel('Count')

# Stars by license
license_stars = df.groupby('license')['stars'].median().sort_values(ascending=False)
license_stars.plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_title('Median Stars by License')
axes[1].set_ylabel('Median Stars')

plt.tight_layout()
plt.show()

## 8. Sample ML Task: Star Prediction

Predict repository star count (log-transformed) from available features.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

# Prepare features
model_df = df.copy()
model_df['log_stars'] = np.log1p(model_df['stars'])
model_df['n_topics'] = model_df['topics'].str.count('\\|') + 1
model_df['days_since_commit'] = (pd.Timestamp('2025-01-01') - pd.to_datetime(model_df['last_commit_date'])).dt.days

# Encode language
le = LabelEncoder()
model_df['language_enc'] = le.fit_transform(model_df['language'])

feature_cols = [
    'forks', 'watchers', 'open_issues', 'closed_issues',
    'contributors', 'commits', 'releases', 'readme_length',
    'has_ci', 'has_code_of_conduct', 'has_contributing_guide',
    'has_wiki', 'has_pages', 'has_discussions',
    'is_archived', 'is_fork', 'size_kb', 'age_days',
    'n_topics', 'days_since_commit', 'language_enc'
]

X = model_df[feature_cols].fillna(0)
y = model_df['log_stars']

# Cross-validation
model = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='r2')

print(f'Star Prediction (log scale) - 5-Fold CV R2: {scores.mean():.3f} (+/- {scores.std():.3f})')

# Feature importance
model.fit(X, y)
feat_imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
feat_imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance for Star Prediction')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual plot
y_pred = model.predict(X)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y, y_pred, alpha=0.2, s=5, c='steelblue')
ax.plot([0, y.max()], [0, y.max()], 'r--', linewidth=1)
ax.set_xlabel('Actual log(stars+1)')
ax.set_ylabel('Predicted log(stars+1)')
ax.set_title(f'Predicted vs Actual Stars (Train R2={r2_score(y, y_pred):.3f})')
plt.tight_layout()
plt.show()

<a id='9'></a>
## 9. Key Insights & Next Steps

### Key Findings

1. **Power-law distribution**: Star counts follow a heavy-tailed distribution -- most repos have <100 stars, but a few have 100K+
2. **Language matters**: Rust and Python repos tend to attract more stars on average
3. **Strong correlations**: Stars, forks, and watchers are highly correlated -- forks are 10-30% of stars
4. **CI adoption**: More popular repos are much more likely to have CI/CD configured
5. **README quality**: Popular repos tend to have significantly longer READMEs -- invest in documentation!
6. **Predictable popularity**: Forks, watchers, and contributors are the best predictors of stars

### Ideas for Using This Dataset

| Project | Complexity | What You Learn |
|---------|------------|----------------|
| EDA + Distribution Analysis | Beginner | Power-law distributions, log transforms |
| Language Trend Visualization | Beginner | pandas, matplotlib, time series |
| Star Prediction (XGBoost) | Intermediate | Regression, feature importance |
| Open Source Health Score | Intermediate | Feature engineering, composite metrics |
| Repository Clustering | Intermediate | K-Means, DBSCAN, UMAP |
| Topic Co-occurrence Network | Advanced | Graph analysis, NetworkX |
| Activity/Archive Prediction | Advanced | Survival analysis, classification |

### Related Resources
- [GitHub Repository Metrics Dataset](https://www.kaggle.com/datasets/lorenzoscaturchio/github-repo-metrics) -- this dataset
- [The State of the Octoverse](https://octoverse.github.com/) -- real GitHub trends to compare against
- [Tabular Playground Series](https://www.kaggle.com/competitions?search=tabular+playground) -- related Kaggle competitions

---

**Dataset by Lorenzo Scaturchio.**

### **If you found this exploration useful, please upvote both the dataset and this notebook! It helps the community discover quality resources.**